# 09 · Validación temporal con una partición aproximada por año calendario — Morbilidad materna severa MIMIC-IV (M2)

| Campo | Contenido |
|---|---|
| Secuencia de notebooks | 01: E1 · E2 · E3 · E4 · P1 → 02: E5 · P2 → 03: E6 · E7 · P3 · R1 · E8 → 04: E8 → 05: E9 (E7) → 06: E9 (E3) → 07: E9 · C1-C3 · R2 · R4 → 08: E9 · C4 → **09: E9 · C4** → 10: P4 · E10 · E11 |
| Qué hace | Desarrolla los modelos solo con admisiones que con certeza son anteriores a un año de corte y los evalúa una vez con admisiones que con certeza son de ese año en adelante (las pacientes del período de desarrollo se retiran de la prueba). Decide el criterio C4 del modelo principal y de los modelos de los retornos R4 y R2, y describe los predictores y el desenlace por período |
| Etapas y puertas | E9 · C4 · E5 (ninguna paciente en ambos períodos) · E11 · P4 |
| Entradas | Base local PostgreSQL de MIMIC-IV (tablas de `sql/`): tablas del notebook 07 (`timing_flags_v3`, `escalation_v3`) |
| Salidas | `data/outputs_<análisis>.txt`, `outputs_temporal_summary.txt`, `outputs_temporal_predictors.txt`, `outputs_cut2017_escalation_reduced*.txt` |
| Uso de los datos | Solo se muestran y guardan resultados agregados; nada a nivel de paciente. No agregar celdas que muestren o exporten filas |
| Tiempo | Cerca de 1–2 horas (un desarrollo y una prueba por análisis y repetición) |
| Siguiente | `10_g4_criteria_fitted_models_es.ipynb` |
| Versión en inglés | `09_temporal_validation_smm.ipynb` (mismo código) |

MIMIC-IV solo informa un `anchor_year_group` de 3 años por paciente, así que el año real de una admisión es un intervalo `[year_lo, year_hi]`. Para un año de corte `C`: desarrollo = `year_hi < C`; prueba = `year_lo >= C`; las admisiones cuyo intervalo cruza `C` se descartan.

| Análisis | Corte | Población |
|---|---|---|
| `cut2017` (principal) | 2017 | partos elegibles |
| `cut2015` (sensibilidad) | 2015 | partos elegibles (corte en el cambio de la CIE-9 a la CIE-10) |
| `cut2017_landmark` | 2017 | población landmark del notebook 07 |
| `cut2017_landmark_labs` | 2017 | landmark con los laboratorios ampliados (R4) |
| `cut2017_escalation` | 2017 | partos elegibles sin UCI antes de T; escalamiento de la atención después de T (R2) |

> **Orden de ejecución.** Correr todas las celdas de arriba hacia abajo. Primero `QUICK = True` como prueba rápida (minutos; nunca se reportan sus cifras) y después `QUICK = False`.

## E1 · Gobierno y configuración — librerías, conexión y configuración

Fija la semilla, los pliegues externos e internos, las repeticiones, las réplicas del bootstrap y los conteos de referencia `EXPECTED` congelados en la puerta P1. `QUICK = True` es una prueba rápida (minutos); sus cifras nunca se reportan.

In [1]:
import os, sys, time, math, warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# notebooks/ -> project root is one level up
sys.path.append(os.path.abspath(os.path.join("..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ----------------------------------------------------------------------------- settings
QUICK        = False         # True = smoke test (small forests, 1 repeat, B = 50). Never report QUICK numbers.
SEED         = 20260101
OUTER_FOLDS  = 5
N_REPEATS    = 3            # repeats of the outer grouped CV (development instability)
INNER_FOLDS  = 3
BOOT         = 1000         # cluster bootstrap replicates for the main analysis
BOOT_SENS    = 300          # replicates for the sensitivity analyses
N_JOBS       = -1
N_TREES      = 300
XGB_TREES    = 200
APPLY_LIMITS = True         # set values outside the plausibility limits of the dictionary to missing
CAT_COLS     = ["ethnicity", "insurance", "language", "admission_type", "admission_location"]
DCA_T        = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20]
EXPECTED     = {"eligible_deliveries": 19383, "smm": 445, "deaths": 3}   # from the audit counts (Q-F / Q-G)

if QUICK:
    OUTER_FOLDS, N_REPEATS, BOOT, BOOT_SENS = 3, 1, 50, 30
    N_TREES, XGB_TREES = 60, 50
print(f"QUICK={QUICK}  outer={OUTER_FOLDS}x{N_REPEATS}  inner={INNER_FOLDS}  B={BOOT}  trees={N_TREES}")

# ----------------------------------------------------------------------------- language and presentation
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
# Nothing below changes a computation: it only decides how tables and messages are shown. The English aggregate
# reports data/outputs_*.txt are written exactly as before (the article cites them); the Spanish notebook also
# writes a translated copy, data/outputs_*_es.txt.
LANG = "es"
import builtins, contextlib, importlib, io, html as _html
import numpy as np
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)
def _tr(s, lang=None):
    """Translate one line keeping its indentation and trailing spaces."""
    lang = lang or LANG
    if lang != "es" or not isinstance(s, str):
        return s
    out = []
    for ln in s.split("\n"):
        core = ln.strip()
        if not core:
            out.append(ln); continue
        i = ln.index(core)
        out.append(ln[:i] + translate(core, "es") + ln[i + len(core):])
    return "\n".join(out)

_REC = None          # while emit() runs: the blocks of the English report, [("text", line) | ("table", frame, index)]

def print(*args, sep=" ", end="\n", file=None, flush=False):
    """Inside emit() the English text goes to the report unchanged; outside it, messages follow LANG."""
    if _REC is not None and file is None:
        _REC.append(("text", sep.join(str(a) for a in args)))
        return builtins.print(*args, sep=sep, end=end, flush=flush)
    if LANG == "es":
        args = [_tr(a, "es") for a in args]
    return builtins.print(*args, sep=sep, end=end, file=file, flush=flush)

def _cell_fmt(v):
    # Display format of a cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        if float(v).is_integer() and abs(v) >= 1000:
            return f"{int(v):,}"
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _html_table(df, index):
    df = df.astype(object)
    df = df.map(_cell_fmt) if hasattr(df, "map") else df.applymap(_cell_fmt)
    return f"<div style='max-height:420px; overflow:auto'>{df.to_html(index=index, na_rep='', border=0)}</div>"

def _in_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False

def table(df, index=False, title=None):
    """Show a table. Inside emit() it also prints the English text of the report (df.to_string)."""
    if isinstance(df, pd.Series):
        df = df.to_frame()
        index = True
    if _REC is not None:
        _REC.append(("table", df, index))
        return builtins.print(df.to_string(index=index))
    if not _in_notebook():
        return builtins.print(translate(df, LANG).to_string(index=index))
    from IPython.display import HTML, display as _display
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>" if title else ""
    _display(HTML(head + _html_table(translate(df, LANG), index)))

def _blocks_html(blocks):
    """Report blocks as HTML: a text line right before a table is its title; rules of '=' are dropped."""
    out, pending = [], []
    def flush_text():
        for ln in pending:
            if ln.strip():
                out.append(f"<div style='white-space:pre-wrap; font-family:monospace; margin:2px 0'>{_html.escape(_tr(ln.rstrip()))}</div>")
        pending.clear()
    for b in blocks:
        if b[0] == "text":
            pending.extend(l for l in b[1].split("\n") if not set(l.strip()) <= {"="} or not l.strip())
        else:
            title = None
            while pending and not pending[-1].strip():
                pending.pop()
            if pending:
                title = pending.pop().strip()
            flush_text()
            if title:
                out.append(f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>")
            out.append(_html_table(translate(b[1], LANG), b[2]))
    flush_text()
    return "\n".join(out)

def _blocks_text_es(blocks):
    """Spanish copy of a report (data/outputs_*_es.txt)."""
    lines = []
    for b in blocks:
        if b[0] == "text":
            lines.extend(_tr(l, "es") for l in b[1].split("\n"))
        else:
            lines.extend(translate(b[1], "es").to_string(index=b[2]).split("\n"))
    return "\n".join(lines) + "\n"

def _emit_to(path, fn, show=True):
    """Run fn, write its English text to `path` exactly as printed and, from the Spanish notebook, a translated copy
    (*_es.txt); show the report as tables."""
    global _REC
    buf = io.StringIO()
    _REC = []
    try:
        with contextlib.redirect_stdout(buf):
            fn()
    finally:
        blocks, _REC = _REC, None
    text = buf.getvalue()
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    if show:
        show_report(blocks)
    if LANG == "es":
        with open(path.replace(".txt", "_es.txt"), "w", encoding="utf-8") as fh:
            fh.write(_blocks_text_es(blocks))
    return text

def show_report(blocks):
    if _in_notebook():
        from IPython.display import HTML, display as _display
        _display(HTML(_blocks_html(blocks)))
    else:
        for b in blocks:
            builtins.print(_tr(b[1]) if b[0] == "text" else translate(b[1], LANG).to_string(index=b[2]))

def notebook_summary(rows, title="Notebook summary"):
    """Summary of the notebook (as in the neonatal repository): one value per row; a value that cannot be computed
    (for example, an analysis that was not run in this session) is shown as n/a."""
    out = {}
    for label, fn in rows:
        try:
            v = fn() if callable(fn) else fn
        except Exception:
            v = "n/a"
        out[label] = v
    table(pd.Series(out, name="value").to_frame(), index=True, title=title)

def _m(res, model, met, d=3):
    """Point value of a metric with its 95% bootstrap interval, from a result with "point" and "boot"."""
    v = f"{res['point'][(model, met)]:.{d}f}"
    b = res.get("boot") or {}
    a = b.get((model, met)) if isinstance(b, dict) else None
    if a is not None and len(a) >= 20:
        lo, hi = np.nanpercentile(a, [2.5, 97.5])
        v += f" ({lo:.{d}f}-{hi:.{d}f})"
    return v

def _cv_auroc(res, model):
    """AUROC averaged over the repeats of a cross-validation result that only keeps out-of-fold predictions."""
    from sklearn.metrics import roc_auc_score
    y = res["y"]
    return float(np.mean([roc_auc_score(y, res["preds"][model][r]) for r in range(len(res["preds"][model]))]))


QUICK=False  outer=5x3  inner=3  B=1000  trees=300


## E7 · Predictores y E4 / P1 — tabla de predictores, diccionario final y conteos de referencia

`predictors_24h_v2` tiene una fila por hospitalización obstétrica con los predictores observados en la ventana (admisión − 24 h, admisión + 24 h). `predictor_dictionary_final_v2` asigna a cada variable su rol (`main model`, `ICU sensitivity only`, `laboratory sensitivity ...`) y sus límites de plausibilidad; los valores fuera de los límites pasan a faltantes. Los datos cargados se comparan con los conteos de referencia; la celda siguiente muestra la decisión de la puerta P1.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Se reproducen los tres conteos | P1 pasa: sigue a los análisis | 19.383 partos, 445 SMM, 3 muertes |
| Algún conteo difiere | Se detiene: revisar las tablas de `sql/` antes de ajustar cualquier modelo | — |

In [2]:
df = pd.read_sql("SELECT * FROM mimiciv.predictors_24h_v2", engine)
dic = pd.read_sql("""SELECT variable, model_role_final, missingness_indicator_main, plausible_low, plausible_high,
                            n_eligible_delivery, coverage_pct
                     FROM mimiciv.predictor_dictionary_final_v2""", engine)

for c in ["eligible_24h", "icu_in_24h", "is_delivery_v2", "smm_any", "died_same_admission"]:
    df[c] = df[c].fillna(False).astype(bool)
df["admittime"] = pd.to_datetime(df["admittime"])

VAR_MAIN = dic.loc[dic.model_role_final == "main model", "variable"].tolist()
VAR_ICU  = dic.loc[dic.model_role_final == "ICU sensitivity only", "variable"].tolist()
VAR_LAB  = dic.loc[dic.model_role_final.str.startswith("laboratory sensitivity"), "variable"].tolist()
IND_MAIN = dic.loc[dic.missingness_indicator_main.fillna(False).astype(bool), "variable"].tolist()

print("Dictionary roles ->", {"main": VAR_MAIN, "ICU only": len(VAR_ICU), "lab sensitivity": len(VAR_LAB),
                              "missingness indicator in main model": IND_MAIN})

# Plausibility limits (proposals pending clinical confirmation): out-of-range values become missing
if APPLY_LIMITS:
    n_set = {}
    for _, r in dic.dropna(subset=["plausible_low", "plausible_high"]).iterrows():
        v = r["variable"]
        if v in df.columns:
            bad = (df[v] < r["plausible_low"]) | (df[v] > r["plausible_high"])
            n_set[v] = int(bad.sum())
            df.loc[bad, v] = np.nan
    print("Values set to missing by plausibility limits (count per variable):", n_set)

# Missingness indicators (computed row by row from the raw value: no information crosses patients)
for v in set(IND_MAIN + VAR_LAB + VAR_ICU):
    df[f"{v}_missing"] = df[v].isna().astype(int)
df["icu_in_24h_bin"] = df["icu_in_24h"].astype(int)
df["y"] = df["smm_any"].astype(int)

# Audit checks against the counts already certified (aggregate only)
elig = df[df.eligible_24h & df.is_delivery_v2]
chk = {"eligible_deliveries": len(elig), "smm": int(elig.smm_any.sum()), "deaths": int(elig.died_same_admission.sum())}
for k, v in chk.items():
    print(f"{k}: {v}  (expected {EXPECTED[k]})  [{'OK' if v == EXPECTED[k] else 'CHECK: differs from the audit counts'}]")
print("Eligible any obstetric admissions:", int(df.eligible_24h.sum()),
      "| SMM:", int(df[df.eligible_24h].smm_any.sum()), "| deaths:", int(df[df.eligible_24h].died_same_admission.sum()))

Roles del diccionario -> {'main': ['hemoglobin', 'creatinine'], 'ICU only': 14, 'lab sensitivity': 11, 'missingness indicator in main model': ['creatinine']}
Valores llevados a faltante por los límites de plausibilidad (conteo por variable): {'hemoglobin': 0, 'hematocrit': 0, 'creatinine': 1}
partos elegibles: 19383  (esperado 19383)  [OK]
SMM: 445  (esperado 445)  [OK]
muertes: 3  (esperado 3)  [OK]
Admisiones obstétricas elegibles de cualquier tipo: 23625 | SMM: 814 | deaths: 5


In [3]:
# Gate G1 decision: the loaded data must reproduce the reference counts frozen at the count audit (aggregates only)
g1 = pd.DataFrame([{"count": k, "loaded": v, "expected": EXPECTED[k],
                    "result": "reproduced" if v == EXPECTED[k] else "DIFFERS"} for k, v in chk.items()])
table(g1, title="Gate G1: reference counts")
table(pd.Series({"G1 decision": "passes: continue to the analyses" if (g1["result"] == "reproduced").all()
                 else "does NOT pass: review the tables of sql/ before fitting any model"}, name="decision"),
      title="Gate G1 decision")

conteo,cargado,esperado,resultado
partos elegibles,"19,383","19,383",reproducido
SMM,445,445,reproducido
muertes,3,3,reproducido


,decisión
decisión P1,pasa: se sigue con los análisis


In [4]:
# Approximate real-year interval and timing class of every eligible delivery hospitalization
tf = pd.read_sql("""SELECT hadm_id, year_lo, year_hi, smm_timing_class
                     FROM mimiciv.timing_flags_v3""", engine)
df = df.drop(columns=[x for x in tf.columns if x != "hadm_id" and x in df.columns]).merge(tf, on="hadm_id", how="left")
df["smm_timing_class"] = df["smm_timing_class"].fillna("not an eligible delivery")
e = df[df.eligible_24h & df.is_delivery_v2]
print(f"Eligible deliveries: {len(e)} (expected {EXPECTED['eligible_deliveries']}); "
      f"with a year interval: {int(e.year_lo.notna().sum())}")

# Escalation of care after T (timed outcome of return R2; sql/07_timing_audit/04_escalation_label.sql)
EXPECTED_ESC = {"population": 19268, "escalation": 101}
esc = pd.read_sql("SELECT hadm_id, escalation_any FROM mimiciv.escalation_v3", engine)
esc["escalation_any"] = esc["escalation_any"].fillna(False).astype(bool)
print(f"escalation population: {len(esc)} (expected {EXPECTED_ESC['population']}); "
      f"events: {int(esc.escalation_any.sum())} (expected {EXPECTED_ESC['escalation']})  "
      f"[{'OK' if (len(esc), int(esc.escalation_any.sum())) == tuple(EXPECTED_ESC.values()) else 'CHECK'}]")
df = df.drop(columns=[x for x in ("escalation_any", "in_esc_pop", "y_esc") if x in df.columns]).merge(esc, on="hadm_id", how="left")
df["in_esc_pop"] = df["escalation_any"].notna()
df["y_esc"] = df["escalation_any"].fillna(False).astype(int)

Partos elegibles: 19383 (esperado 19383); con intervalo de año: 19383
población de escalamiento: 19268 (esperado 19268); eventos: 101 (esperado 101)  [OK]


## E9 · C4 · E5 — particiones temporales

`make_temporal(name)` devuelve los conjuntos de desarrollo y prueba de un análisis; la tabla siguiente da los conteos de cada partición (esperados para `cut2017` según la consulta Q-T4: 11.104 admisiones / 209 SMM antes, 6.315 / 201 desde 2017, 1.964 que cruzan; los conteos de prueba mostrados son menores porque se retiran las admisiones de prueba de pacientes que ya están en desarrollo). `TEMPORAL_RUN` en la sección siguiente lista los análisis que se corren.

In [5]:
TEMPORAL = {
    "cut2017":          dict(cut=2017, landmark=False),
    "cut2015":          dict(cut=2015, landmark=False),
    "cut2017_landmark": dict(cut=2017, landmark=True),
    # second gate-G4 classification (criterion C4 for the models of the returns)
    "cut2017_landmark_labs": dict(cut=2017, landmark=True, labs=True),   # return R4: extended laboratory set
    "cut2017_escalation":    dict(cut=2017, landmark=False, escalation=True),  # return R2: escalation after T
}
EXPECTED_Q_T4 = {2017: (11104, 209, 6315, 201, 1964), 2015: (8017, 121, 8279, 236, 3087)}

def make_temporal(name):
    cfg = TEMPORAL[name]; C = cfg["cut"]
    d = df[df.eligible_24h & df.is_delivery_v2].copy()
    label = "y"
    if cfg.get("escalation"):
        d = d[d.in_esc_pop & ~d.icu_in_24h]
        label = "y_esc"
    if cfg["landmark"]:
        d = d[~d.icu_in_24h & ~d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"])]
    before, after = d.year_hi < C, d.year_lo >= C
    dev, test_all = d[before], d[after]
    shared = test_all.subject_id.isin(set(dev.subject_id))
    test = test_all[~shared]
    num  = [c for c in dict.fromkeys(["age_at_admission"] + VAR_MAIN) if c in d.columns]
    binc = [c for c in dict.fromkeys(f"{v}_missing" for v in IND_MAIN) if c in d.columns and dev[c].nunique() > 1]
    if cfg.get("labs"):
        num  = list(dict.fromkeys(num + [v for v in VAR_LAB if v in d.columns]))
        binc = list(dict.fromkeys(binc + [f"{v}_missing" for v in VAR_LAB
                                          if f"{v}_missing" in d.columns and dev[f"{v}_missing"].nunique() > 1]))
    cat  = [c for c in CAT_COLS if c in d.columns]
    info = dict(cut=C, population=len(d), dev_adm=len(dev), dev_pat=dev.subject_id.nunique(), dev_events=int(dev[label].sum()),
                straddling=int((~before & ~after).sum()), straddling_events=int(d[~before & ~after][label].sum()),
                test_adm_all=len(test_all), test_events_all=int(test_all[label].sum()),
                test_removed_shared=int(shared.sum()), test_removed_shared_events=int(test_all[shared][label].sum()),
                test_adm=len(test), test_pat=test.subject_id.nunique(), test_events=int(test[label].sum()))
    return dict(name=name, train=dev.reset_index(drop=True), test=test.reset_index(drop=True),
                num=num, bin=binc, cat=cat, label=label, info=info)

rows = []
for nm in TEMPORAL:
    i = make_temporal(nm)["info"]
    rows.append({"analysis": nm, "cut": i["cut"],
                 "dev adm (SMM)": f"{i['dev_adm']} ({i['dev_events']})",
                 "dev prev %": round(100 * i["dev_events"] / max(i["dev_adm"], 1), 2),
                 "straddling (SMM)": f"{i['straddling']} ({i['straddling_events']})",
                 "test before patient removal (SMM)": f"{i['test_adm_all']} ({i['test_events_all']})",
                 "removed: patient in dev (SMM)": f"{i['test_removed_shared']} ({i['test_removed_shared_events']})",
                 "test adm (SMM)": f"{i['test_adm']} ({i['test_events']})",
                 "test prev %": round(100 * i["test_events"] / max(i["test_adm"], 1), 2)})
    if not TEMPORAL[nm]["landmark"] and not TEMPORAL[nm].get("escalation"):
        exp = EXPECTED_Q_T4[i["cut"]]
        got = (i["dev_adm"], i["dev_events"], i["test_adm_all"], i["test_events_all"], i["straddling"])
        print(f"{nm}: counts vs Q-T4 {got} expected {exp}  [{'OK' if got == exp else 'CHECK'}]")
table(pd.DataFrame(rows))

cut2017: conteos frente a Q-T4 (11104, 209, 6315, 201, 1964), esperados (11104, 209, 6315, 201, 1964)  [OK]
cut2015: conteos frente a Q-T4 (8017, 121, 8279, 236, 3087), esperados (8017, 121, 8279, 236, 3087)  [OK]


análisis,cut,dev adm (SMM),dev prev %,straddling (SMM),test before patient removal (SMM),removed: patient in dev (SMM),test adm (SMM),test prev %
cut2017,"2,017",11104 (209),1.88,1964 (35),6315 (201),1240 (28),5075 (173),3.41
cut2015,"2,015",8017 (121),1.51,3087 (88),8279 (236),1104 (22),7175 (214),2.98
cut2017_landmark,"2,017",11008 (127),1.15,1948 (21),6242 (137),1225 (20),5017 (117),2.33
cut2017_landmark_labs,"2,017",11008 (127),1.15,1948 (21),6242 (137),1225 (20),5017 (117),2.33
cut2017_escalation,"2,017",11032 (46),0.42,1956 (8),6280 (47),1234 (6),5046 (41),0.81


## E8 · Modelado, validación y métricas — preprocesamiento, algoritmos, ajuste anidado, ensamble y calibración

Todo lo que aprende de los datos queda dentro de la parte de entrenamiento de cada partición (puerta P2: ninguna paciente a ambos lados):

* `Winsor` recorta cada columna numérica a los percentiles 0,5–99,5 *de los datos de entrenamiento*.
* La imputación por la mediana, el escalado y la codificación one-hot se ajustan dentro de un `Pipeline`, una vez por ajuste.
* Los hiperparámetros se eligen con **validación cruzada interna agrupada por paciente** (precisión promedio de las predicciones fuera de muestra).
* El meta-modelo del ensamble se entrena con las predicciones **fuera de muestra** de los algoritmos base (nunca con predicciones dentro de muestra).
* El calibrador de Platt se entrena con predicciones fuera de muestra de las pacientes de entrenamiento; las pacientes de prueba externa solo se evalúan.

In [6]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def expit(z):
    return 1.0 / (1.0 + np.exp(-z))

class Winsor(BaseEstimator, TransformerMixin):
    """Clip each column to training percentiles (fit on the training data only)."""
    def __init__(self, lo=0.5, hi=99.5):
        self.lo = lo; self.hi = hi
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, self.lo, axis=0)
        self.hi_ = np.nanpercentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def make_prep(num, binc, cat):
    parts = [("num", Pipeline([("w", Winsor()),
                               ("imp", SimpleImputer(strategy="median", keep_empty_features=True)),
                               ("sc", StandardScaler())]), num)]
    if binc:
        parts.append(("bin", "passthrough", binc))
    if cat:
        parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                       ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                            min_frequency=0.01, sparse_output=False))]), cat))
    return ColumnTransformer(parts)

def prep_frame(d, num, binc, cat):
    X = d[num + binc + cat].copy()
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
    return X

# Small prespecified grids (class weights are options; no resampling)
GRID = {
    # The grids cover values beyond those most often chosen in preliminary runs, so that the selected value
    # is not at the edge of the grid (ET depth, RF depth and XGB depth; class weights remain options).
    "LR":  [dict(C=C, cw=cw) for C in (0.01, 0.1, 1.0) for cw in (None, "balanced")],
    "RF":  [dict(depth=dp, cw=cw) for dp in (2, 4, 8) for cw in (None, "balanced_subsample")],
    "ET":  [dict(depth=dp, cw=cw) for dp in (4, 8, 12) for cw in (None, "balanced_subsample")],
    "XGB": [dict(depth=dp, spw=sp) for dp in (1, 2, 4) for sp in (1, "sqrt")],
}
MODELS = ["LR", "RF", "ET", "XGB", "Stack", "Stack_raw"]
BOOT_MODELS = ["LR", "RF", "ET", "XGB", "Stack"]
PAIRS = [("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "LR"), ("Stack", "XGB")]

def cfg_str(c):
    return ",".join(f"{k}={v}" for k, v in c.items())

def make_clf(name, cfg, ytr, seed):
    if name == "LR":
        return LogisticRegression(C=cfg["C"], class_weight=cfg["cw"], max_iter=3000)
    if name == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                      class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "ET":
        return ExtraTreesClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                    class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "XGB":
        pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
        spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
        return XGBClassifier(n_estimators=XGB_TREES, learning_rate=0.05, max_depth=cfg["depth"], subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw, eval_metric="logloss",
                             tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)
    raise ValueError(name)

def fit_predict(name, cfg, prep, Xtr, ytr, Xte, seed):
    pipe = Pipeline([("prep", clone(prep)), ("clf", make_clf(name, cfg, ytr, seed))])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

def platt(o, t, ytr):
    c = LogisticRegression(penalty=None, max_iter=1000).fit(logit(o).reshape(-1, 1), ytr)
    return c.predict_proba(logit(t).reshape(-1, 1))[:, 1]

def tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, seed):
    """Inside one outer training set: tune, build out-of-fold predictions, stack, calibrate; score the outer test set."""
    inner = list(StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed).split(Xtr, ytr, gtr))
    names = list(GRID)
    chosen, oof, pred_te = {}, {}, {}
    for name in names:
        best = None
        for cfg in GRID[name]:
            o = np.zeros(len(ytr))
            for tr, va in inner:
                o[va] = fit_predict(name, cfg, prep, Xtr.iloc[tr], ytr[tr], Xtr.iloc[va], seed)
            ap = average_precision_score(ytr, o)
            if best is None or ap > best[0]:
                best = (ap, cfg, o)
        chosen[name] = best[1]; oof[name] = best[2]
        pred_te[name] = fit_predict(name, best[1], prep, Xtr, ytr, Xte, seed)
    # explicit stacking: logistic meta-learner on out-of-fold base predictions (C = 0.1 as in the original framework)
    Z_oof = np.column_stack([logit(oof[n]) for n in names])
    Z_te = np.column_stack([logit(pred_te[n]) for n in names])
    meta = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof, ytr)
    stack_te_raw = meta.predict_proba(Z_te)[:, 1]
    # cross-fitted meta out-of-fold predictions -> calibrator never sees the predictions of its own training rows
    mo = np.zeros(len(ytr))
    for tr, va in StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed + 1).split(Z_oof, ytr, gtr):
        m = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof[tr], ytr[tr])
        mo[va] = m.predict_proba(Z_oof[va])[:, 1]
    out = {n: platt(oof[n], pred_te[n], ytr) for n in names}
    out["Stack"] = platt(mo, stack_te_raw, ytr)
    out["Stack_raw"] = stack_te_raw
    return out, chosen

def run_cv(spec, label=translate("", LANG)):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    y = d[spec["label"]].values.astype(int)
    g = d["subject_id"].values
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    n = len(y)
    preds = {m: np.full((N_REPEATS, n), np.nan) for m in MODELS}
    chosen_log = []
    split_log = []
    t0 = time.time(); done = 0; total = N_REPEATS * OUTER_FOLDS
    for r in range(N_REPEATS):
        outer = StratifiedGroupKFold(OUTER_FOLDS, shuffle=True, random_state=SEED + r)
        for k, (tr, te) in enumerate(outer.split(X, y, g)):
            assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
            split_log.append(dict(repeat=r + 1, fold=k + 1,
                                  train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                                  train_event_pat=len(set(g[tr][y[tr] == 1])),
                                  test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()),
                                  test_event_pat=len(set(g[te][y[te] == 1])), shared_patients=0))
            out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, SEED + 100 * r + k)
            for m in MODELS:
                preds[m][r, te] = out[m]
            for nme, cfg in chosen.items():
                chosen_log.append((nme, cfg_str(cfg)))
            done += 1
            if done == 1:
                el = time.time() - t0
                print(f"[{spec['name']}] first outer fold took {el/60:.1f} min -> projected total about {el*total/60:.0f} min")
    print(f"[{spec['name']}] cross-validation finished in {(time.time()-t0)/60:.1f} min; "
          f"every admission scored exactly once per repeat: {bool(np.isfinite(preds['Stack']).all())}")
    return dict(spec=spec, y=y, g=g, preds=preds, chosen=chosen_log, X=X, prep=prep, splits=pd.DataFrame(split_log))

## E8 · Modelado, validación y métricas — métricas, bootstrap por conglomerados y diferencias pareadas

La discriminación, la calibración y la utilidad clínica se calculan con las predicciones fuera de muestra de cada repetición y se promedian entre repeticiones. El bootstrap remuestrea **pacientes** (todas las admisiones de una paciente se mueven juntas), así que los intervalos reflejan las admisiones repetidas. Las diferencias entre modelos usan las **mismas** muestras bootstrap (pareadas).

In [7]:
def citl(y, z, w):
    """Calibration-in-the-large: intercept with the slope fixed at 1 (weighted Newton)."""
    a = 0.0
    for _ in range(40):
        pr = expit(z + a)
        g = np.sum(w * (y - pr)); h = np.sum(w * pr * (1 - pr))
        if h < 1e-12:
            break
        step = g / h; a += step
        if abs(step) < 1e-9:
            break
    return a

def calib_slope(y, z, w):
    """Calibration slope (weighted logistic recalibration, Newton)."""
    a, b = 0.0, 1.0
    for _ in range(40):
        pr = expit(a + b * z)
        r = w * (y - pr); v = w * pr * (1 - pr)
        gr = np.array([r.sum(), (r * z).sum()])
        H = np.array([[v.sum(), (v * z).sum()], [(v * z).sum(), (v * z * z).sum()]])
        try:
            dlt = np.linalg.solve(H, gr)
        except np.linalg.LinAlgError:
            break
        a += dlt[0]; b += dlt[1]
        if np.abs(dlt).max() < 1e-9:
            break
    return b

def top_threshold(p, w, k):
    order = np.argsort(-p)
    cw = np.cumsum(w[order])
    idx = int(np.searchsorted(cw, k * cw[-1]))
    return p[order][min(idx, len(p) - 1)]

def metrics_one(y, p, w):
    z = logit(p)
    m = {}
    m["auroc"] = roc_auc_score(y, p, sample_weight=w)
    m["auprc"] = average_precision_score(y, p, sample_weight=w)
    m["brier"] = float(np.average((p - y) ** 2, weights=w))
    m["slope"] = calib_slope(y, z, w)
    m["citl"] = citl(y, z, w)
    m["oe"] = float(np.sum(w * y) / np.sum(w * p))
    edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
    bins = np.searchsorted(edges, p)
    W = w.sum(); ece = 0.0
    for b in range(10):
        mk = bins == b; wb = w[mk].sum()
        if wb > 0:
            ece += wb / W * abs(np.sum(w[mk] * y[mk]) / wb - np.sum(w[mk] * p[mk]) / wb)
    m["ece"] = ece
    ev = np.sum(w * y)
    for k in (0.05, 0.10):
        sel = p >= top_threshold(p, w, k)
        m[f"sens_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / ev
        m[f"ppv_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
    for pt in DCA_T:
        sel = p >= pt
        tp = np.sum(w[sel] * y[sel]); fp = np.sum(w[sel] * (1 - y[sel]))
        m[f"nb_{pt}"] = tp / W - fp / W * pt / (1 - pt)
    return m

def all_metrics(y, preds, w, models, strata):
    """Metrics for each model, averaged over repeats. Returns dict[(model, metric)] and the per-repeat list."""
    per = []
    for r in range(N_REPEATS):
        cur = {}
        for mname in models:
            for k, v in metrics_one(y, preds[mname][r], w).items():
                cur[(mname, k)] = v
        prev = np.sum(w * y) / w.sum()
        for pt in DCA_T:
            cur[("TreatAll", f"nb_{pt}")] = prev - (1 - prev) * pt / (1 - pt)
        for sname, mask in strata.items():
            for mname in ("LR", "Stack"):
                if mname not in models:
                    continue
                wm = w[mask]; ym = y[mask]; pm = preds[mname][r][mask]
                if np.sum(wm * ym) >= 1 and np.sum(wm * (1 - ym)) >= 1:
                    cur[(mname, f"auroc_{sname}")] = roc_auc_score(ym, pm, sample_weight=wm)
                    cur[(mname, f"auprc_{sname}")] = average_precision_score(ym, pm, sample_weight=wm)
                else:
                    cur[(mname, f"auroc_{sname}")] = np.nan
                    cur[(mname, f"auprc_{sname}")] = np.nan
        per.append(cur)
    keys = per[0].keys()
    mean = {k: float(np.nanmean([c[k] for c in per])) if not all(np.isnan(c[k]) for c in per) else np.nan for k in keys}
    return mean, per

def bootstrap(res, B, strata, seed):
    y, g, preds = res["y"], res["g"], res["preds"]
    ug, inv = np.unique(g, return_inverse=True)
    G = len(ug)
    rng = np.random.default_rng(seed)
    out = {}
    t0 = time.time()
    for b in range(B):
        cnt = np.bincount(rng.integers(0, G, G), minlength=G)
        w = cnt[inv].astype(float)
        if np.sum(w * y) < 1:
            continue
        mean, _ = all_metrics(y.astype(float), preds, w, BOOT_MODELS, strata)
        for k, v in mean.items():
            out.setdefault(k, []).append(v)
        for a_, b_ in PAIRS:            # paired differences (same bootstrap sample)
            for met in ("auroc", "auprc", "brier") + tuple(f"nb_{pt}" for pt in DCA_T):
                out.setdefault((f"{a_}-{b_}", met), []).append(mean[(a_, met)] - mean[(b_, met)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return {k: np.array(v) for k, v in out.items()}

## E8 · E11 Reporte — función de reporte

Arma el reporte agregado de un análisis: discriminación, calibración, diferencias pareadas entre modelos, eventos captados en el 5% / 10% de mayor riesgo, beneficio neto, inestabilidad entre repeticiones y los hiperparámetros elegidos en los ciclos internos. `emit()` lo guarda como `data/outputs_<análisis>.txt` (en inglés, como lo cita el artículo) y lo muestra como tablas; el notebook en español también escribe `data/outputs_<análisis>_es.txt`.

In [8]:
def _ci(boot, key, d=3):
    a = boot.get(key)
    if a is None or len(a) < 20:
        return ""
    lo, hi = np.nanpercentile(a, [2.5, 97.5])
    return f" ({lo:.{d}f}-{hi:.{d}f})"

def decile_table(res, models=("LR", "Stack")):
    y = res["y"].astype(float)
    out = None
    for m in models:
        acc = []
        for r in range(N_REPEATS):
            p = res["preds"][m][r]
            edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
            b = np.searchsorted(edges, p)
            acc.append(pd.DataFrame({"b": b, "p": p, "y": y}).groupby("b").agg(n=("y", "size"), pred=("p", "mean"), obs=("y", "mean")))
        mean = sum(acc) / len(acc)
        t = pd.DataFrame({"decile": mean.index + 1, "n": mean["n"].round(0).astype(int)}) if out is None else out
        t[f"{m}_pred"] = mean["pred"].round(4).values
        t[f"{m}_obs"] = mean["obs"].round(4).values
        out = t
    return out

def plot_calibration(res):
    import matplotlib.pyplot as plt
    t = decile_table(res)
    fig, ax = plt.subplots(figsize=(5, 5))
    mx = float(max(t["LR_pred"].max(), t["LR_obs"].max(), t["Stack_pred"].max(), t["Stack_obs"].max())) * 1.1
    ax.plot([0, mx], [0, mx], color="grey", lw=1, ls="--", label=translate("ideal", LANG))
    ax.plot(t["LR_pred"], t["LR_obs"], "o-", label=translate("Logistic regression", LANG))
    ax.plot(t["Stack_pred"], t["Stack_obs"], "s-", label=translate("Stacked ensemble", LANG))
    ax.set_xlabel(translate("Mean predicted risk (decile)", LANG)); ax.set_ylabel(translate("Observed SMM rate", LANG))
    ax.set_title(translate(f"Calibration by decile -- {res['spec']['name']}", LANG)); ax.legend(); plt.show()   # displayed only, not saved

def emit(name, fn):
    """Write the aggregate report data/outputs_<name>.txt (English text, as cited by the article) and show it as tables;
    the Spanish notebook also writes data/outputs_<name>_es.txt. Aggregates only."""
    path = f"../data/outputs_{name}.txt"
    text = _emit_to(path, fn)
    print(f"[aggregate report saved as data/{os.path.basename(path)}  ({len(text)} characters)]")

def run_analysis(name, B):
    spec = make_spec(name)
    res = run_cv(spec)
    d = spec["data"]
    strata = {}
    if name == "main":
        strata = {"icu1": d["icu_in_24h"].values.astype(bool), "icu0": ~d["icu_in_24h"].values.astype(bool)}
    y = res["y"].astype(float)
    res["point"], res["per"] = all_metrics(y, res["preds"], np.ones(len(y)), MODELS + [], strata)
    res["boot"] = bootstrap(res, B, strata, SEED + 7)
    res["strata"] = strata
    return res

def report(res):
    sp = res["spec"]; d = sp["data"]; y = res["y"]
    pt, per, bt = res["point"], res["per"], res["boot"]
    n, ev, npat = len(y), int(y.sum()), d.subject_id.nunique()
    print("=" * 110)
    print(f"ANALYSIS {sp['name']}: {n} admissions, {npat} patients, {ev} events ({100*ev/n:.2f}%), "
          f"numeric={len(sp['num'])}, binary={len(sp['bin'])}, categorical={len(sp['cat'])}")
    print(f"Repeats={N_REPEATS}, outer folds={OUTER_FOLDS}, inner folds={INNER_FOLDS}, bootstrap B={len(next(iter(bt.values()))) if bt else 0}, QUICK={QUICK}")
    print("=" * 110)

    sl = res["splits"]
    print("\n[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)")
    table(sl[sl.repeat == 1].drop(columns=["repeat"]))
    print(f"    Across all {len(sl)} splits: test patients per fold {sl.test_pat.min()}-{sl.test_pat.max()}, "
          f"event-bearing test patients per fold {sl.test_event_pat.min()}-{sl.test_event_pat.max()}, "
          f"patients present on both sides: {int(sl.shared_patients.sum())}")

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["sd_auroc_repeats"] = f"{np.std([c[(m,'auroc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        r["sd_auprc_repeats"] = f"{np.std([c[(m,'auprc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        rows.append(r)
    print("\n[1] Discrimination and overall accuracy (mean over repeats; 95% cluster-bootstrap CI)")
    print(f"    prevalence = {y.mean():.4f}  (AUPRC of a non-informative model equals the prevalence)")
    table(pd.DataFrame(rows))

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("slope", 2), ("citl", 2), ("oe", 2), ("ece", 4)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        rows.append(r)
    print("\n[2] Calibration (slope ideal 1; calibration-in-the-large ideal 0; O/E ideal 1; ECE with 10 quantile bins)")
    table(pd.DataFrame(rows))

    rows = []
    for a_, b_ in PAIRS:
        r = {"comparison (first minus second)": f"{a_} - {b_}"}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(a_, met)] - pt[(b_, met)]:+.{dd}f}" + _ci(bt, (f"{a_}-{b_}", met), dd)
        rows.append(r)
    print("\n[3] Paired differences (same bootstrap samples); the last row compares the ensemble with XGBoost")
    table(pd.DataFrame(rows))

    rows = []
    for m in BOOT_MODELS:
        rows.append({"model": m,
                     "sens_top5%": f"{pt[(m,'sens_top5')]:.3f}" + _ci(bt, (m, "sens_top5")),
                     "ppv_top5%": f"{pt[(m,'ppv_top5')]:.3f}",
                     "sens_top10%": f"{pt[(m,'sens_top10')]:.3f}" + _ci(bt, (m, "sens_top10")),
                     "ppv_top10%": f"{pt[(m,'ppv_top10')]:.3f}"})
    print("\n[4] Events captured among the 5% / 10% of admissions with the highest predicted risk")
    table(pd.DataFrame(rows))

    rows = []
    for m in BOOT_MODELS + ["TreatAll"]:
        rows.append({"strategy": m, **{f"pt={t:g}": round(pt[(m, f'nb_{t}')], 5) for t in DCA_T}})
    rows.append({"strategy": "TreatNone", **{f"pt={t:g}": 0.0 for t in DCA_T}})
    print("\n[5] Net benefit (decision curve) at the stated threshold probabilities")
    table(pd.DataFrame(rows))
    rows = []
    for a_, b_ in [("Stack", "LR"), ("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "XGB")]:
        rows.append({"comparison": f"{a_} - {b_}", **{f"pt={t:g}": f"{pt[(a_,f'nb_{t}')]-pt[(b_,f'nb_{t}')]:+.5f}" + _ci(bt, (f"{a_}-{b_}", f"nb_{t}"), 5) for t in DCA_T[:6]}})
    print("    Paired net-benefit differences:")
    table(pd.DataFrame(rows))

    if res["strata"]:
        rows = []
        for sname, lab in [("icu1", "ICU admission within 24 h"), ("icu0", "no ICU within 24 h")]:
            mk = res["strata"][sname]
            r = {"stratum": lab, "n": int(mk.sum()), "events": int(y[mk].sum())}
            for m in ("LR", "Stack"):
                r[f"{m}_auroc"] = (f"{pt[(m, 'auroc_' + sname)]:.3f}" + _ci(bt, (m, "auroc_" + sname))) if not np.isnan(pt[(m, 'auroc_' + sname)]) else "n/a"
                r[f"{m}_auprc"] = (f"{pt[(m, 'auprc_' + sname)]:.3f}") if not np.isnan(pt[(m, 'auprc_' + sname)]) else "n/a"
            rows.append(r)
        print("\n[6] Performance by ICU admission within the first 24 h (same predictions, stratified evaluation)")
        table(pd.DataFrame(rows))

    print("\n[8] Calibration by decile of predicted risk (outer-test predictions; mean over repeats)")
    table(decile_table(res))

    ch = pd.DataFrame(res["chosen"], columns=["learner", "config"])
    tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
    print(f"\n[7] Hyperparameters selected in the inner loops (out of {N_REPEATS*OUTER_FOLDS} outer fits per learner)")
    table(tab)

## E9 · C4 — desarrollar en el período anterior, evaluar una vez en el posterior

`run_temporal` desarrolla todo el pipeline con el conjunto de desarrollo (`tune_stack_calibrate`, como en los notebooks 03 y 07) y evalúa una vez el conjunto de prueba; `report_temporal` muestra discriminación, calibración, diferencias pareadas, 5% / 10% superior, beneficio neto y calibración por decil.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| AUROC temporal ≥ 0,70 y pendiente de calibración entre 0,90 y 1,10 | C4 cumple | — |
| Pendiente fuera de 0,90–1,10 o AUROC temporal < 0,70 | C4 no cumple | Modelo principal: AUROC 0,747, pendiente 0,83 (0,71–0,97), O/E 1,34 |

In [9]:
def run_temporal(spec):
    tr, te = spec["train"], spec["test"]
    Xtr = prep_frame(tr, spec["num"], spec["bin"], spec["cat"])
    Xte = prep_frame(te, spec["num"], spec["bin"], spec["cat"])
    ytr = tr[spec["label"]].values.astype(int); gtr = tr["subject_id"].values
    yte = te[spec["label"]].values.astype(int); gte = te["subject_id"].values
    assert not set(gtr) & set(gte), "patient present in development and test"
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    preds = {m: np.full((N_REPEATS, len(yte)), np.nan) for m in MODELS}
    chosen = []
    t0 = time.time()
    for r in range(N_REPEATS):
        out, ch = tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, SEED + 1000 * r)
        for m in MODELS:
            preds[m][r] = out[m]
        chosen += [(n, cfg_str(c)) for n, c in ch.items()]
        print(f"[{spec['name']}] repeat {r + 1}/{N_REPEATS} done ({(time.time() - t0) / 60:.1f} min)")
    res = dict(spec=spec, y=yte, g=gte, preds=preds, chosen=chosen, ytr=ytr)
    res["point"], res["per"] = all_metrics(yte.astype(float), preds, np.ones(len(yte)), MODELS, {})
    res["boot"] = bootstrap(res, BOOT_SENS, {}, SEED + 7)
    return res

def report_temporal(res):
    sp, i = res["spec"], res["spec"]["info"]
    pt, per, bt, y = res["point"], res["per"], res["boot"], res["y"]
    print("=" * 110)
    print(f"TEMPORAL VALIDATION {sp['name']} (cut {i['cut']}), QUICK={QUICK}, repeats={N_REPEATS}, bootstrap B={BOOT_SENS}")
    print("=" * 110)
    print(f"Development (year_hi < {i['cut']}): {i['dev_adm']} admissions, {i['dev_pat']} patients, {i['dev_events']} events "
          f"(prevalence {100 * i['dev_events'] / i['dev_adm']:.2f}%)")
    print(f"Discarded (interval straddles {i['cut']}): {i['straddling']} admissions, {i['straddling_events']} events")
    print(f"Test (year_lo >= {i['cut']}): {i['test_adm_all']} admissions; removed because the patient is in development: "
          f"{i['test_removed_shared']} ({i['test_removed_shared_events']} events)")
    print(f"Test analysed: {i['test_adm']} admissions, {i['test_pat']} patients, {i['test_events']} events "
          f"(prevalence {100 * i['test_events'] / i['test_adm']:.2f}%)")

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["sd_auroc_repeats"] = f"{np.std([c[(m, 'auroc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        rows.append(r)
    print("\n[1] Discrimination in the later period (mean over repeats; 95% patient-level bootstrap CI)")
    table(pd.DataFrame(rows))

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("slope", 2), ("citl", 2), ("oe", 2), ("ece", 4)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["mean predicted"] = f"{np.mean(res['preds'][m]):.4f}"
        rows.append(r)
    print(f"\n[2] Calibration in the later period (observed prevalence {y.mean():.4f}; O/E > 1 = risk underestimated)")
    table(pd.DataFrame(rows))

    rows = []
    for a_, b_ in PAIRS:
        r = {"comparison": f"{a_} - {b_}"}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(a_, met)] - pt[(b_, met)]:+.{dd}f}" + _ci(bt, (f"{a_}-{b_}", met), dd)
        rows.append(r)
    print("\n[3] Paired differences (same bootstrap samples)")
    table(pd.DataFrame(rows))

    rows = []
    for m in BOOT_MODELS:
        rows.append({"model": m, "sens_top5%": f"{pt[(m, 'sens_top5')]:.3f}" + _ci(bt, (m, "sens_top5")),
                     "ppv_top5%": f"{pt[(m, 'ppv_top5')]:.3f}",
                     "sens_top10%": f"{pt[(m, 'sens_top10')]:.3f}" + _ci(bt, (m, "sens_top10")),
                     "ppv_top10%": f"{pt[(m, 'ppv_top10')]:.3f}"})
    print("\n[4] Events captured among the 5% / 10% of admissions with the highest predicted risk")
    table(pd.DataFrame(rows))

    rows = [{"strategy": m, **{f"pt={t:g}": round(pt[(m, f'nb_{t}')], 5) for t in DCA_T}} for m in BOOT_MODELS + ["TreatAll"]]
    print("\n[5] Net benefit by threshold probability")
    table(pd.DataFrame(rows))

    print("\n[6] Calibration by decile of predicted risk (mean over repeats)")
    table(decile_table(res))

    ch = pd.DataFrame(res["chosen"], columns=["model", "config"]).value_counts().rename("times chosen").reset_index()
    print("\n[7] Hyperparameters chosen in the inner loops of the development set")
    table(ch.sort_values(["model", "times chosen"], ascending=[True, False]))

In [10]:
TEMPORAL_RUN = list(TEMPORAL)   # remove names to run only some analyses
RESULTS = {}
for nm in TEMPORAL_RUN:
    sp = make_temporal(nm)
    for side in ("train", "test"):
        ev = int(sp[side][sp["label"]].sum())
        if ev < 40:
            print(f"WARNING: {nm} has only {ev} events in the {side} set; report as exploratory.")
    RESULTS[nm] = run_temporal(sp)
    emit(nm, lambda r=RESULTS[nm]: report_temporal(r))

[cut2017] repetición 1/3 terminada (0.6 min)
[cut2017] repetición 2/3 terminada (1.2 min)
[cut2017] repetición 3/3 terminada (1.7 min)
  bootstrap: 5 réplicas tardaron 0 s -> proyectado 0.4 min para B=300


[reporte agregado guardado como data/outputs_cut2017.txt  (4950 caracteres)]
[cut2015] repetición 1/3 terminada (0.5 min)
[cut2015] repetición 2/3 terminada (1.0 min)
[cut2015] repetición 3/3 terminada (1.5 min)
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 0.5 min para B=300


[reporte agregado guardado como data/outputs_cut2015.txt  (4993 caracteres)]
[cut2017_landmark] repetición 1/3 terminada (0.6 min)
[cut2017_landmark] repetición 2/3 terminada (1.2 min)
[cut2017_landmark] repetición 3/3 terminada (1.8 min)
  bootstrap: 5 réplicas tardaron 0 s -> proyectado 0.4 min para B=300


[reporte agregado guardado como data/outputs_cut2017_landmark.txt  (5003 caracteres)]
[cut2017_landmark_labs] repetición 1/3 terminada (0.7 min)
[cut2017_landmark_labs] repetición 2/3 terminada (1.3 min)
[cut2017_landmark_labs] repetición 3/3 terminada (1.9 min)
  bootstrap: 5 réplicas tardaron 0 s -> proyectado 0.4 min para B=300


[reporte agregado guardado como data/outputs_cut2017_landmark_labs.txt  (4958 caracteres)]
[cut2017_escalation] repetición 1/3 terminada (0.6 min)
[cut2017_escalation] repetición 2/3 terminada (1.3 min)
[cut2017_escalation] repetición 3/3 terminada (1.9 min)
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 0.5 min para B=300


[reporte agregado guardado como data/outputs_cut2017_escalation.txt  (5008 caracteres)]


In [11]:
_RESULTS_MAIN = dict(RESULTS)   # copy of the main temporal results, used only by the notebook summary

## E11 · P4 — tabla resumen para el artículo

Una fila por análisis y modelo (regresión logística, XGBoost, ensamble apilado): tamaños de desarrollo y prueba, prevalencia de cada período, AUROC y AUPRC con intervalos del 95%, pendiente de calibración y O/E en el período posterior. Se guarda como `data/outputs_temporal_summary.txt`.

In [12]:
def temporal_summary():
    rows = []
    for nm, res in RESULTS.items():
        i, pt, bt = res["spec"]["info"], res["point"], res["boot"]
        for m in ("LR", "XGB", "Stack"):
            rows.append({"analysis": nm, "model": m,
                         "dev (events, prev %)": f"{i['dev_adm']} ({i['dev_events']}, {100 * i['dev_events'] / i['dev_adm']:.2f})",
                         "test (events, prev %)": f"{i['test_adm']} ({i['test_events']}, {100 * i['test_events'] / i['test_adm']:.2f})",
                         "auroc": f"{pt[(m, 'auroc')]:.3f}" + _ci(bt, (m, "auroc")),
                         "auprc": f"{pt[(m, 'auprc')]:.3f}" + _ci(bt, (m, "auprc")),
                         "slope": f"{pt[(m, 'slope')]:.2f}" + _ci(bt, (m, "slope"), 2),
                         "O/E": f"{pt[(m, 'oe')]:.2f}" + _ci(bt, (m, "oe"), 2)})
    print(f"TEMPORAL VALIDATION SUMMARY (QUICK={QUICK})")
    table(pd.DataFrame(rows))

emit("temporal_summary", temporal_summary)

análisis,modelo,"desarrollo (eventos, prev. %)","prueba (eventos, prev. %)",AUROC,AUPRC,pendiente,O/E
cut2017,LR,"11104 (209, 1.88)","5075 (173, 3.41)",0.744 (0.705-0.778),0.154 (0.114-0.208),0.79 (0.68-0.92),1.27 (1.09-1.46)
cut2017,XGB,"11104 (209, 1.88)","5075 (173, 3.41)",0.747 (0.707-0.783),0.146 (0.111-0.205),0.83 (0.71-0.97),1.34 (1.15-1.52)
cut2017,Stack,"11104 (209, 1.88)","5075 (173, 3.41)",0.752 (0.713-0.789),0.169 (0.126-0.229),0.84 (0.72-0.98),1.31 (1.12-1.49)
cut2015,LR,"8017 (121, 1.51)","7175 (214, 2.98)",0.764 (0.734-0.794),0.135 (0.104-0.185),0.90 (0.80-1.03),1.41 (1.23-1.60)
cut2015,XGB,"8017 (121, 1.51)","7175 (214, 2.98)",0.760 (0.724-0.792),0.120 (0.096-0.164),0.92 (0.82-1.05),1.48 (1.30-1.68)
cut2015,Stack,"8017 (121, 1.51)","7175 (214, 2.98)",0.766 (0.731-0.798),0.133 (0.105-0.181),0.90 (0.79-1.02),1.42 (1.24-1.60)
cut2017_landmark,LR,"11008 (127, 1.15)","5017 (117, 2.33)",0.659 (0.614-0.705),0.053 (0.035-0.092),0.78 (0.53-1.01),1.61 (1.37-1.91)
cut2017_landmark,XGB,"11008 (127, 1.15)","5017 (117, 2.33)",0.670 (0.627-0.713),0.048 (0.038-0.071),0.82 (0.58-1.04),1.67 (1.41-1.98)
cut2017_landmark,Stack,"11008 (127, 1.15)","5017 (117, 2.33)",0.666 (0.620-0.715),0.052 (0.039-0.080),0.83 (0.58-1.09),1.64 (1.39-1.96)
cut2017_landmark_labs,LR,"11008 (127, 1.15)","5017 (117, 2.33)",0.680 (0.636-0.724),0.047 (0.037-0.067),0.80 (0.55-1.08),2.02 (1.68-2.37)


[reporte agregado guardado como data/outputs_temporal_summary.txt  (2362 caracteres)]


## E9 · C4 — predictores y desenlace por período (ítem 20c de TRIPOD+AI)

Compara el período de desarrollo (con certeza antes de 2017) con el de prueba (con certeza desde 2017, sin las pacientes del período de desarrollo), como en `cut2017`. Los conteos entre 1 y 9 se muestran como `<10` y las categorías raras se agrupan para que ningún conteo pequeño se pueda recuperar por resta. Se guarda como `data/outputs_temporal_predictors.txt`.

In [13]:
def temporal_predictors():
    sp = make_temporal("cut2017")
    parts = {"development (<2017)": sp["train"], "test (>=2017)": sp["test"]}
    def cnt(n, d):
        return "<10" if 0 < n < 10 else f"{n:,} ({100 * n / d:.1f})"
    rows = [{"variable": "Admissions, n", **{k: f"{len(v):,}" for k, v in parts.items()}},
            {"variable": "SMM, n (%)", **{k: cnt(int(v['y'].sum()), len(v)) for k, v in parts.items()}}]
    for v in dict.fromkeys(["age_at_admission"] + VAR_MAIN):
        if v not in sp["train"].columns:
            continue
        r_med = {"variable": f"{v}, median (IQR) among measured"}
        r_pct = {"variable": f"{v}, measured, n (%)"}
        for k, d in parts.items():
            x = d[v].dropna()
            r_med[k] = f"{x.median():.1f} ({x.quantile(.25):.1f}-{x.quantile(.75):.1f})" if len(x) >= 10 else "<10"
            r_pct[k] = cnt(len(x), len(d))
        rows.append(r_med)
        if v != "age_at_admission":
            rows.append(r_pct)
    for c in CAT_COLS:
        if c not in sp["train"].columns:
            continue
        lab = {k: d[c].fillna("missing").astype(str) for k, d in parts.items()}
        cats = pd.concat(lab.values()).value_counts().index.tolist()
        small = [g for g in cats if min((lab[k] == g).sum() for k in parts) < 10]
        for k in parts:
            lab[k] = lab[k].where(~lab[k].isin(small), "other (grouped)")
        if small and min((lab[k] == "other (grouped)").sum() for k in parts) < 10:
            # the grouped cell is still small: merge it with the smallest remaining category
            rest = [g for g in cats if g not in small]
            if rest:
                tgt = min(rest, key=lambda g: sum((lab[k] == g).sum() for k in parts))
                for k in parts:
                    lab[k] = lab[k].replace({tgt: "other (grouped)"})
        for g in pd.concat(lab.values()).value_counts().index:
            rows.append({"variable": f"{c}: {g}", **{k: cnt(int((lab[k] == g).sum()), len(parts[k])) for k in parts}})
    t = pd.DataFrame(rows)
    print("PREDICTORS AND OUTCOME BY PERIOD, cut 2017 (development vs test as analysed in cut2017)")
    print("Values are n (%) unless stated; counts 1-9 shown as <10; rare categories grouped.")
    table(t)

emit("temporal_predictors", temporal_predictors)

variable,desarrollo (<2017),prueba (>=2017)
"Admissions, n","11,104","5,075"
"SMM, n (%)",209 (1.9),173 (3.4)
"age_at_admission, median (IQR) among measured",32.0 (28.0-36.0),33.0 (29.0-36.0)
"hemoglobin, median (IQR) among measured",12.2 (11.4-13.0),11.9 (11.0-12.7)
"hemoglobin, measured, n (%)","9,071 (81.7)","4,110 (81.0)"
"creatinine, median (IQR) among measured",0.6 (0.5-0.7),0.6 (0.5-0.7)
"creatinine, measured, n (%)","2,016 (18.2)","1,325 (26.1)"
ethnicity: WHITE,"4,890 (44.0)","2,026 (39.9)"
ethnicity: BLACK/AFRICAN AMERICAN,"1,940 (17.5)",804 (15.8)
ethnicity: OTHER,731 (6.6),480 (9.5)


[reporte agregado guardado como data/outputs_temporal_predictors.txt  (4818 caracteres)]


## C4 · reducción por tamaño de muestra — criterio C4 del modelo de escalamiento reducido

Regla fijada antes de esta corrida (`docs/decision_G3_reduction_2026-11-09.md`). La misma partición temporal de `cut2017_escalation` con el conjunto reducido de predictores fijado de antemano. C4 cumple si el AUROC temporal es 0,70 o más y el intervalo del 95% de la pendiente de calibración incluye 1. Se guarda como `data/outputs_cut2017_escalation_reduced.txt`.

In [14]:
RESULTS = {}

def add_reduced_columns(frame):
    """Columns of the prespecified reduced set (docs/decision_G3_reduction_2026-11-09.md). Row-wise, no outcome information."""
    eth = frame["ethnicity"].fillna("").astype(str).str.upper()
    frame["race5"] = np.select(
        [eth.str.startswith("WHITE"), eth.str.startswith("BLACK"),
         eth.str.startswith("HISPANIC") | eth.str.startswith("SOUTH AMERICAN"), eth.str.startswith("ASIAN")],
        ["White", "Black", "Hispanic or Latina", "Asian"], default="Other or unknown")
    frame["ins_medicaid"] = (frame["insurance"].astype(str) == "Medicaid").astype(int)
    frame["lang_english"] = (frame["language"].astype(str) == "English").astype(int)
    frame["adm_urgent"] = (frame["admission_type"].astype(str) == "URGENT").astype(int)
    frame["loc_physician"] = (frame["admission_location"].astype(str) == "PHYSICIAN REFERRAL").astype(int)
    return frame

RED_BIN = ["ins_medicaid", "lang_english", "adm_urgent", "loc_physician"]
RED_CAT = ["race5"]
P_MAX_REDUCTION = 16

df = add_reduced_columns(df)

def make_temporal_reduced():
    sp = make_temporal("cut2017_escalation")
    sp["train"] = add_reduced_columns(sp["train"].copy()); sp["test"] = add_reduced_columns(sp["test"].copy())
    sp["bin"] = [b for b in (["creatinine_missing"] + RED_BIN) if b in sp["train"].columns and sp["train"][b].nunique() > 1]
    sp["cat"] = list(RED_CAT)
    sp["name"] = "cut2017_escalation_reduced"
    return sp

sp_red = make_temporal_reduced()
X_ = prep_frame(sp_red["train"], sp_red["num"], sp_red["bin"], sp_red["cat"])
P_RED_T = int(make_prep(sp_red["num"], sp_red["bin"], sp_red["cat"]).fit(X_).transform(X_).shape[1])
print(f"Reduced set in the development period: {P_RED_T} parameters (maximum allowed {P_MAX_REDUCTION})")
assert P_RED_T <= P_MAX_REDUCTION

RESULTS["cut2017_escalation_reduced"] = run_temporal(sp_red)

def reduced_temporal_report():
    res = RESULTS["cut2017_escalation_reduced"]
    report_temporal(res)
    pt, bt = res["point"], res["boot"]
    print("\nCRITERION C4 (reduced escalation model): temporal AUROC >= 0.70 and 95% interval of the slope includes 1")
    for m in ("LR", "XGB", "Stack"):
        lo, hi = np.nanpercentile(bt[(m, "slope")], [2.5, 97.5])
        verdict = "neither (AUROC < 0.65)" if pt[(m, "auroc")] < 0.65 else (
            "met" if (pt[(m, "auroc")] >= 0.70 and lo <= 1 <= hi) else "not met")
        print(f"  {m:5s} AUROC {pt[(m, 'auroc')]:.3f}{_ci(bt, (m, 'auroc'))}  slope {pt[(m, 'slope')]:.2f} ({lo:.2f}-{hi:.2f})  "
              f"O/E {pt[(m, 'oe')]:.2f}{_ci(bt, (m, 'oe'), 2)}  -> C4 {verdict}")

emit("cut2017_escalation_reduced", reduced_temporal_report)

Conjunto reducido en el período de desarrollo: 13 parámetros (máximo permitido 16)
[cut2017_escalation_reduced] repetición 1/3 terminada (0.7 min)
[cut2017_escalation_reduced] repetición 2/3 terminada (1.3 min)
[cut2017_escalation_reduced] repetición 3/3 terminada (1.8 min)
  bootstrap: 5 réplicas tardaron 0 s -> proyectado 0.4 min para B=300


[reporte agregado guardado como data/outputs_cut2017_escalation_reduced.txt  (5298 caracteres)]


## Exploratorio (post hoc) — malla de XGBoost ampliada, criterio C4 del modelo de escalamiento reducido

La misma malla ampliada del notebook 07, sección 16; ajustada solo dentro del período de desarrollo. Se guarda como `data/outputs_cut2017_escalation_reduced_xgbgrid.txt`.

In [15]:
GRID_XGB_SMALL = list(GRID["XGB"])
GRID_XGB_WIDE = [dict(depth=dp, spw=sp, lr=lr, mcw=mcw)
                 for dp in (1, 2, 3) for sp in (1, "sqrt") for lr in (0.02, 0.05, 0.1) for mcw in (5, 20)]
_make_clf_base = globals().get("_make_clf_base", make_clf)

def make_clf(name, cfg, ytr, seed):
    if name != "XGB" or "lr" not in cfg:
        return _make_clf_base(name, cfg, ytr, seed)
    pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
    spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
    return XGBClassifier(n_estimators=XGB_TREES, learning_rate=cfg["lr"], max_depth=cfg["depth"], subsample=0.8,
                         colsample_bytree=0.8, min_child_weight=cfg["mcw"], scale_pos_weight=spw, eval_metric="logloss",
                         tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)

GRID["XGB"] = GRID_XGB_WIDE
try:
    sp_x = make_temporal_reduced(); sp_x["name"] = "cut2017_escalation_reduced_xgbgrid"
    RESULTS["cut2017_escalation_reduced_xgbgrid"] = run_temporal(sp_x)
finally:
    GRID["XGB"] = GRID_XGB_SMALL

def xgbgrid_temporal_report():
    res = RESULTS["cut2017_escalation_reduced_xgbgrid"]
    report_temporal(res)
    pt, bt = res["point"], res["boot"]
    print("\nEXPLORATORY C4 (wider XGBoost grid): temporal AUROC >= 0.70 and 95% interval of the slope includes 1")
    for m in ("LR", "XGB", "Stack"):
        lo, hi = np.nanpercentile(bt[(m, "slope")], [2.5, 97.5])
        ok = pt[(m, "auroc")] >= 0.70 and lo <= 1 <= hi
        print(f"  {m:5s} AUROC {pt[(m, 'auroc')]:.3f}{_ci(bt, (m, 'auroc'))}  slope {pt[(m, 'slope')]:.2f} ({lo:.2f}-{hi:.2f})  -> C4 {'met' if ok else 'not met'}")

emit("cut2017_escalation_reduced_xgbgrid", xgbgrid_temporal_report)

[cut2017_escalation_reduced_xgbgrid] repetición 1/3 terminada (0.8 min)
[cut2017_escalation_reduced_xgbgrid] repetición 2/3 terminada (1.5 min)
[cut2017_escalation_reduced_xgbgrid] repetición 3/3 terminada (2.4 min)
  bootstrap: 5 réplicas tardaron 0 s -> proyectado 0.4 min para B=300


[reporte agregado guardado como data/outputs_cut2017_escalation_reduced_xgbgrid.txt  (5350 caracteres)]


## Resumen del notebook

Validación temporal: desarrollo antes del corte y una sola evaluación después (criterio C4).

In [16]:
# Summary of notebook 09
_ALL = {**globals().get("_RESULTS_MAIN", {}), **RESULTS}
_rows = []
for _nm in ["cut2017", "cut2015", "cut2017_landmark", "cut2017_escalation_reduced"]:
    _rows.append((f"{_nm}: AUROC XGBoost (95% CI)", lambda nm=_nm: _m(_ALL[nm], "XGB", "auroc")))
    _rows.append((f"{_nm}: calibration slope XGBoost (95% CI)", lambda nm=_nm: _m(_ALL[nm], "XGB", "slope", 2)))
    _rows.append((f"{_nm}: observed/expected XGBoost (95% CI)", lambda nm=_nm: _m(_ALL[nm], "XGB", "oe", 2)))
notebook_summary(_rows)

,valor
cut2017: AUROC XGBoost (IC 95%),0.747 (0.707-0.783)
cut2017: pendiente de calibración XGBoost (IC 95%),0.83 (0.71-0.97)
cut2017: observados/esperados XGBoost (IC 95%),1.34 (1.15-1.52)
cut2015: AUROC XGBoost (IC 95%),0.760 (0.724-0.792)
cut2015: pendiente de calibración XGBoost (IC 95%),0.92 (0.82-1.05)
cut2015: observados/esperados XGBoost (IC 95%),1.48 (1.30-1.68)
cut2017_landmark: AUROC XGBoost (IC 95%),0.670 (0.627-0.713)
cut2017_landmark: pendiente de calibración XGBoost (IC 95%),0.82 (0.58-1.04)
cut2017_landmark: observados/esperados XGBoost (IC 95%),1.67 (1.41-1.98)
cut2017_escalation_reduced: AUROC XGBoost (IC 95%),0.699 (0.620-0.778)
